# time-stage-instrumentation — faded example 3: Crash-safe stage context manager

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `time-stage-instrumentation`. The last cell reports your progress on the `Logging: time-stage instrumentation` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: time-stage instrumentation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`time-stage-instrumentation`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "time-stage-instrumentation"
DD_SUBTOPIC = "Logging: time-stage instrumentation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `@contextmanager` timer records start time at entry and, in a `finally` block, the elapsed time at exit. Putting the record in `finally` makes it exception-safe: the timing is logged even if the wrapped block raises, then the exception continues to propagate.

## Faded exercise 3

### Make the stage timer exception-safe

Implement the `stage(name, acc)` context manager. It records `t0` at entry and must record the elapsed time into `acc[name]` (using `acc.get(name, 0.0)`) in a way that survives an exception inside the `with` block. Complete the `finally` body that computes and accumulates the elapsed time.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import time
import contextlib

@contextlib.contextmanager
def stage(name, acc):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        acc[name] = acc.get(name, 0.0) + (time.perf_counter() - t0)

acc = {}
with stage('x', acc):
    time.sleep(0.003)
print(acc)


def _test():
    acc = {}
    with stage('load', acc):
        time.sleep(0.005)
    assert acc['load'] >= 0.005, acc
    # exception inside the block still records, then propagates
    raised = False
    try:
        with stage('train', acc):
            time.sleep(0.003)
            raise ValueError('boom')
    except ValueError:
        raised = True
    assert raised, 'exception should propagate'
    assert 'train' in acc and acc['train'] >= 0.003, acc
    # nested re-entry into same key sums both
    acc2 = {}
    with stage('s', acc2):
        time.sleep(0.002)
    with stage('s', acc2):
        time.sleep(0.002)
    assert acc2['s'] >= 0.004, acc2


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import time
import contextlib

@contextlib.contextmanager
def stage(name, acc):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        acc[name] = acc.get(name, 0.0) + (time.perf_counter() - t0)

acc = {}
with stage('x', acc):
    time.sleep(0.003)
print(acc)
```
</details>